# **Facebook Scraper Notebook for HealthPH+**


# **Dependencies**

In [ ]:
import requests
import time
import pandas as pd
from datetime import datetime
from urllib.parse import urlparse, parse_qs
import os
import re


In [ ]:
from pathlib import Path
from typing import Any
from urllib.parse import quote_plus
import hashlib
import json

try:
    from playwright.async_api import (
        async_playwright,
        TimeoutError as PlaywrightTimeoutError,
    )
except ImportError as exc:
    raise ImportError(
        "playwright is not installed. Run: pip install playwright && playwright install chromium"
    ) from exc

In [ ]:
print("✅ Libraries loaded successfully")

## Facebook

Notebook-first proof of concept for public Facebook scraping using Playwright Async API (Jupyter-safe).
This section writes pipeline-compatible raw CSV rows with standard fields (`created_at`, `id`, `text`, `source`) and legacy Facebook fields (`timestamp`, `post_id`, `message`) for current merger compatibility.


In [ ]:
from pathlib import Path
from typing import Any
from urllib.parse import urljoin
import hashlib
import re
import sys

try:
    from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeoutError
except ImportError as exc:
    raise ImportError(
        "playwright is not installed. Run: pip install playwright && playwright install chromium"
    ) from exc


In [ ]:
# Facebook scraper config (Playwright)
FB_TARGETS = [
    # Public page handles or full URLs
    "DepartmentofHealthPH",
]

FB_KEYWORDS = [
    "lagnat",
    "ubo",
    "sipon",
    "pneumonia",
    "tb",
    "covid",
]

FB_MAX_POSTS_PER_TARGET = 50
FB_SCROLL_ROUNDS = 20
FB_SCROLL_WAIT_MS = 1500
FB_HEADLESS = True
FB_NAV_TIMEOUT_MS = 60000
FB_START_DATE = pd.Timestamp("2025-01-01", tz="UTC")
FB_END_DATE = pd.Timestamp.now(tz="UTC")

FB_OUTPUT_DIR = Path("../data/raw/facebook")
FB_OUTPUT_FILE = FB_OUTPUT_DIR / f"facebook_notebook_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.csv"

print("Targets:", FB_TARGETS)
print("Keywords:", ", ".join(FB_KEYWORDS))
print("Date range:", FB_START_DATE, "to", FB_END_DATE)
print("Output:", FB_OUTPUT_FILE)


In [ ]:
def _safe_text(value: Any) -> str:
    if value is None:
        return ""
    return str(value).strip()


def _to_utc_datetime(value: Any) -> pd.Timestamp | None:
    if value is None or value == "":
        return None
    ts = pd.to_datetime(value, errors="coerce", utc=True)
    if pd.isna(ts):
        return None
    return ts


def _contains_keyword(text: str, keywords: list[str]) -> bool:
    hay = text.lower()
    return any(keyword.lower() in hay for keyword in keywords)


def _target_to_url(target: str) -> str:
    target = _safe_text(target)
    if target.startswith("http://") or target.startswith("https://"):
        return target
    return f"https://m.facebook.com/{target}"


def _extract_post_id(post_url: str, text: str, created_at: str) -> str:
    patterns = [
        r"/posts/(\d+)",
        r"[?&]story_fbid=(\d+)",
        r"[?&]fbid=(\d+)",
        r"/videos/(\d+)",
        r"/reel/(\d+)",
    ]
    for pattern in patterns:
        match = re.search(pattern, post_url)
        if match:
            return match.group(1)

    base = f"{post_url}|{created_at}|{text[:120]}"
    return hashlib.sha1(base.encode("utf-8")).hexdigest()


async def _parse_article_timestamp(article) -> pd.Timestamp | None:
    try:
        abbr = await article.query_selector("abbr")
        if abbr:
            utime = await abbr.get_attribute("data-utime")
            if utime:
                ts = pd.to_datetime(pd.to_numeric(utime, errors="coerce"), unit="s", errors="coerce", utc=True)
                if not pd.isna(ts):
                    return ts

            title = await abbr.get_attribute("title")
            parsed_title = _to_utc_datetime(title)
            if parsed_title is not None:
                return parsed_title

            text_value = _safe_text(await abbr.inner_text())
            parsed_text = _to_utc_datetime(text_value)
            if parsed_text is not None:
                return parsed_text

        time_el = await article.query_selector("time")
        if time_el:
            dt_attr = await time_el.get_attribute("datetime")
            parsed_dt = _to_utc_datetime(dt_attr)
            if parsed_dt is not None:
                return parsed_dt
    except Exception:
        return None

    return None


async def _extract_article_text(article) -> str:
    text_selectors = [
        "div[data-ad-preview=message]",
        "div[dir=auto]",
    ]

    chunks = []
    for selector in text_selectors:
        nodes = await article.query_selector_all(selector)
        for node in nodes:
            value = _safe_text(await node.inner_text())
            if value:
                chunks.append(value)

    if not chunks:
        fallback = _safe_text(await article.inner_text())
        return fallback

    combined = " ".join(chunks)
    return re.sub(r"\s+", " ", combined).strip()


async def _extract_post_url(article, page_url: str) -> str:
    link_selectors = [
        "a[href*='/posts/']",
        "a[href*='story.php']",
        "a[href*='/videos/']",
        "a[href*='/reel/']",
        "a[href*='permalink']",
    ]

    for selector in link_selectors:
        link = await article.query_selector(selector)
        if not link:
            continue
        href = await link.get_attribute("href")
        if href:
            return urljoin(page_url, href)

    return page_url


async def _normalize_facebook_article(article, page_url: str, keywords: list[str]) -> dict[str, Any] | None:
    text = await _extract_article_text(article)
    if not text:
        return None
    if keywords and not _contains_keyword(text, keywords):
        return None

    post_url = await _extract_post_url(article, page_url=page_url)
    post_time = await _parse_article_timestamp(article)
    created_at = "" if post_time is None else post_time.strftime("%Y-%m-%d %H:%M:%S UTC")
    post_id = _extract_post_id(post_url=post_url, text=text, created_at=created_at)

    return {
        "created_at": created_at,
        "id": post_id,
        "text": text,
        "source": "facebook",
        # Legacy columns expected by current merger logic
        "timestamp": created_at,
        "post_id": post_id,
        "message": text,
        # Optional field kept for compatibility with existing raw samples
        "reactions_count": None,
    }


async def scrape_facebook_targets(
    targets: list[str],
    keywords: list[str],
    max_posts_per_target: int = 50,
    scroll_rounds: int = 20,
    scroll_wait_ms: int = 1500,
    headless: bool = True,
    nav_timeout_ms: int = 60000,
    start_date: pd.Timestamp | None = None,
    end_date: pd.Timestamp | None = None,
) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []

    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=headless)
        context = await browser.new_context()
        page = await context.new_page()
        page.set_default_timeout(nav_timeout_ms)

        for target in targets:
            target_url = _target_to_url(target)
            print(f"\nScraping target: {target} -> {target_url}")

            try:
                await page.goto(target_url, wait_until="domcontentloaded")
            except PlaywrightTimeoutError:
                print(f"  ! Timeout opening {target_url}")
                continue
            except Exception as exc:
                print(f"  ! Failed to open {target_url}: {exc}")
                continue

            kept = 0
            seen_ids: set[str] = set()

            for _ in range(scroll_rounds):
                articles = await page.query_selector_all("div[role=article]")
                if not articles:
                    # Fallback for mobile/basic layouts
                    articles = await page.query_selector_all("article")

                for article in articles:
                    normalized = await _normalize_facebook_article(article, page_url=page.url, keywords=keywords)
                    if normalized is None:
                        continue

                    if normalized["id"] in seen_ids:
                        continue

                    post_time = _to_utc_datetime(normalized["created_at"])
                    if start_date is not None and post_time is not None and post_time < start_date:
                        continue
                    if end_date is not None and post_time is not None and post_time > end_date:
                        continue

                    rows.append(normalized)
                    seen_ids.add(normalized["id"])
                    kept += 1

                    if kept >= max_posts_per_target:
                        break

                if kept >= max_posts_per_target:
                    break

                await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
                await page.wait_for_timeout(scroll_wait_ms)

            print(f"  + Kept {kept} post(s)")

        await context.close()
        await browser.close()

    columns = [
        "created_at",
        "id",
        "text",
        "source",
        "timestamp",
        "post_id",
        "message",
        "reactions_count",
    ]

    if not rows:
        return pd.DataFrame(columns=columns)

    return pd.DataFrame(rows, columns=columns)


In [ ]:
facebook_df = await scrape_facebook_targets(
    targets=FB_TARGETS,
    keywords=FB_KEYWORDS,
    max_posts_per_target=FB_MAX_POSTS_PER_TARGET,
    scroll_rounds=FB_SCROLL_ROUNDS,
    scroll_wait_ms=FB_SCROLL_WAIT_MS,
    headless=FB_HEADLESS,
    nav_timeout_ms=FB_NAV_TIMEOUT_MS,
    start_date=FB_START_DATE,
    end_date=FB_END_DATE,
)

print("Rows collected:", len(facebook_df))
facebook_df.head()


In [ ]:
FB_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
facebook_df.to_csv(FB_OUTPUT_FILE, index=False, encoding="utf-8-sig")
print(f"Saved {len(facebook_df)} row(s) -> {FB_OUTPUT_FILE}")


In [ ]:
required_cols = ["created_at", "id", "text", "source"]
legacy_cols = ["timestamp", "post_id", "message"]

missing_cols = [col for col in required_cols if col not in facebook_df.columns]
missing_legacy_cols = [col for col in legacy_cols if col not in facebook_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")
if missing_legacy_cols:
    raise ValueError(f"Missing merger-compat columns: {missing_legacy_cols}")

validation_summary = pd.DataFrame({
    "metric": [
        "rows",
        "missing_created_at",
        "missing_id",
        "missing_text",
        "duplicate_id_rows",
        "missing_timestamp",
        "missing_post_id",
        "missing_message",
    ],
    "value": [
        int(len(facebook_df)),
        int(facebook_df["created_at"].isna().sum() + (facebook_df["created_at"].astype(str).str.strip() == "").sum()),
        int(facebook_df["id"].isna().sum() + (facebook_df["id"].astype(str).str.strip() == "").sum()),
        int(facebook_df["text"].isna().sum() + (facebook_df["text"].astype(str).str.strip() == "").sum()),
        int(facebook_df.duplicated(subset=["id"]).sum()),
        int(facebook_df["timestamp"].isna().sum() + (facebook_df["timestamp"].astype(str).str.strip() == "").sum()),
        int(facebook_df["post_id"].isna().sum() + (facebook_df["post_id"].astype(str).str.strip() == "").sum()),
        int(facebook_df["message"].isna().sum() + (facebook_df["message"].astype(str).str.strip() == "").sum()),
    ],
})

validation_summary


In [ ]:
# Optional smoke test: verify current merger can read facebook raw files
repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from modules.data_merger import load_source

facebook_merged_preview = load_source("facebook", data_root="../data/raw")
print("Merged Facebook rows available:", len(facebook_merged_preview))
facebook_merged_preview.head(3)
